# 01. Business context and data audit

**Objective.** Find out what is actually in `application_train.csv` before choosing any technique.

**Business question.** Can this table support a probability-of-default model, and what must be fixed or avoided first?

**Why this analysis is needed.** Every later choice (imputation, transformation, features, split) has to point back to a problem found here. Nothing in this notebook is decided in advance.

**Scope.** Only the application table is audited here. The bureau, previous-application and instalment tables have not been loaded yet, so the history features are still open.

*Educational project. Not a lending policy.*

## 1. Size, duplicates and identifiers

**Why:** duplicates would inflate any test score, and an ID that correlates with the outcome would be a leak.

In [1]:
import sys
sys.path.append("..")

import numpy as np
import pandas as pd
from scipy import stats

from src import config
from src.data import load_applications, load_column_descriptions

df = load_applications()
print("shape:", df.shape)
print("duplicate rows:", df.duplicated().sum())
print("duplicate IDs:", df[config.ID_COLUMN].duplicated().sum())
print("Spearman(ID, TARGET):", round(stats.spearmanr(df[config.ID_COLUMN], df["TARGET"])[0], 4))

shape: (307511, 122)
duplicate rows: 0
duplicate IDs: 0
Spearman(ID, TARGET): -0.0021


**Result and interpretation.** One row per applicant, no duplicates, and the ID has essentially no rank correlation with the target (about -0.002), so row order carries no hidden signal.

**Decision.** `SK_ID_CURR` is an identifier with no meaning for a new applicant. It is kept only as a join key and excluded from model features.

## 2. What is being predicted, and how common is it?

**Why:** the metric, the imbalance handling and the wording of every result depend on the target definition.

In [2]:
print(df["TARGET"].value_counts())
print(f"positive rate: {df['TARGET'].mean():.2%}")
descriptions = load_column_descriptions()
target_row = descriptions[descriptions["Row"] == "TARGET"].iloc[0]
print(target_row["Description"])

TARGET
0    282686
1     24825
Name: count, dtype: int64
positive rate: 8.07%
Target variable (1 - client with payment difficulties: he/she had late payment more than X days on at least one of the first Y installments of the loan in our sample, 0 - all other cases)


**Result and interpretation.** 24,825 of 307,511 applicants are positive (8.07%). The data owner defines the label as a late payment of more than X days on at least one of the first Y instalments, and does not disclose X or Y. It is therefore an early-repayment-difficulty label, not lifetime default.

**Decision.**
- Results are worded as "probability of early payment difficulty" and called PD only with that caveat.
- Accuracy is uninformative here: predicting "no difficulty" for everyone scores about 92%. Ranking and probability metrics will be used instead (decided formally in Phase 6).
- Whether 8% is imbalanced enough to need resampling is a modelling question, tested later, not assumed now.

## 3. Missing values

**Why:** 67 of 122 columns have gaps. The right treatment depends on why the values are missing, so that comes first.

In [3]:
missing = df.isna().mean().sort_values(ascending=False)
building = [c for c in df.columns if c.endswith(("_AVG", "_MODE", "_MEDI"))]
print("columns with any missing:", (missing > 0).sum())
print("columns missing in more than half the rows:", (missing > 0.5).sum())
print("building-description columns:", len(building))
print("applicants missing every building column:", round(df[building].isna().all(axis=1).mean(), 3))
print()
print(missing[~missing.index.isin(building)][missing > 0].round(3).head(8))

columns with any missing: 67
columns missing in more than half the rows: 41
building-description columns: 47
applicants missing every building column: 0.474

OWN_CAR_AGE                   0.660
EXT_SOURCE_1                  0.564
OCCUPATION_TYPE               0.313
EXT_SOURCE_3                  0.198
AMT_REQ_CREDIT_BUREAU_HOUR    0.135
AMT_REQ_CREDIT_BUREAU_WEEK    0.135
AMT_REQ_CREDIT_BUREAU_MON     0.135
AMT_REQ_CREDIT_BUREAU_YEAR    0.135
dtype: float64


In [4]:
print("OWN_CAR_AGE missing, by FLAG_OWN_CAR:")
print(df.groupby("FLAG_OWN_CAR")["OWN_CAR_AGE"].apply(lambda s: s.isna().mean()))

is_pensioner = df["NAME_INCOME_TYPE"] == "Pensioner"
print("OCCUPATION_TYPE missing | pensioner:", df.loc[is_pensioner, "OCCUPATION_TYPE"].isna().mean().round(3))
print("OCCUPATION_TYPE missing | other:", df.loc[~is_pensioner, "OCCUPATION_TYPE"].isna().mean().round(3))

rows = []
for col in ["EXT_SOURCE_1", "EXT_SOURCE_3", "AMT_REQ_CREDIT_BUREAU_YEAR", "COMMONAREA_AVG"]:
    absent = df[col].isna()
    rows.append({
        "column": col,
        "rows_missing": absent.sum(),
        "default_if_missing": df.loc[absent, "TARGET"].mean(),
        "default_if_present": df.loc[~absent, "TARGET"].mean(),
    })
print(pd.DataFrame(rows).round(4).to_string(index=False))

OWN_CAR_AGE missing, by FLAG_OWN_CAR:
FLAG_OWN_CAR
N    1.000000
Y    0.000048
Name: OWN_CAR_AGE, dtype: float64
OCCUPATION_TYPE missing | pensioner: 1.0
OCCUPATION_TYPE missing | other: 0.163
                    column  rows_missing  default_if_missing  default_if_present
              EXT_SOURCE_1        173378              0.0852              0.0750
              EXT_SOURCE_3         60965              0.0931              0.0777
AMT_REQ_CREDIT_BUREAU_YEAR         41519              0.1034              0.0772
            COMMONAREA_AVG        214865              0.0857              0.0691


**Result and interpretation.**
- 41 columns are missing in more than half the rows. All but two of them (`OWN_CAR_AGE` and `EXT_SOURCE_1`) belong to the 47 building-description columns, which are average, mode and median versions of the same measurements; 47.4% of applicants have none of them.
- Some gaps are structural, not random: `OWN_CAR_AGE` is missing for every applicant without a car and for almost none with one, and `OCCUPATION_TYPE` is missing for every pensioner.
- Other gaps carry signal. Applicants with `EXT_SOURCE_1` missing default at 8.52% versus 7.50% when present; for `AMT_REQ_CREDIT_BUREAU_YEAR` it is 10.34% versus 7.72%. Missingness alone is associated with risk.

**Decision (nothing is imputed here).**
- Filling everything with the median would erase the structural and informative patterns above, so it is not applied.
- Tree models can take missing values directly. For the logistic-regression baseline, median filling plus a missing-indicator will be tried on columns where missingness is informative, fitted on training data only. This is decided and compared in Phases 5 and 6.
- No column is dropped for being sparse alone. Whether the near-duplicate building columns can be reduced without loss is tested in Phase 5.

## 4. A suspicious value in `DAYS_EMPLOYED`

**Why:** `DAYS_EMPLOYED` should be a negative number of days before the application. The summary statistics looked wrong (mean 63,815, maximum 365,243).

In [5]:
placeholder = df["DAYS_EMPLOYED"] == 365243
print("rows with 365243:", placeholder.sum(), f"({placeholder.mean():.2%})")
print(df.loc[placeholder, "NAME_INCOME_TYPE"].value_counts())
print("default rate, placeholder rows:", round(df.loc[placeholder, "TARGET"].mean(), 4))
print("default rate, all other rows:", round(df.loc[~placeholder, "TARGET"].mean(), 4))
print("ORGANIZATION_TYPE == 'XNA' on exactly the same rows:",
      ((df["ORGANIZATION_TYPE"] == "XNA") == placeholder).all())
print("largest value among the other rows:", df.loc[~placeholder, "DAYS_EMPLOYED"].max(), "days")

rows with 365243: 55374 (18.01%)
NAME_INCOME_TYPE
Pensioner     55352
Unemployed       22
Name: count, dtype: int64
default rate, placeholder rows: 0.054
default rate, all other rows: 0.0866
ORGANIZATION_TYPE == 'XNA' on exactly the same rows: True
largest value among the other rows: 0 days


**Result and interpretation.** 55,374 applicants (18.0%) share the value 365,243, about 1,000 years, which is not a real employment length. Almost all are pensioners, and these rows also carry `ORGANIZATION_TYPE = XNA`. They form a distinct group with a lower default rate (5.40% versus 8.66%). The value looks like a code for "not applicable", not a measurement.

**Decision.** Replace 365,243 with missing and add a `DAYS_EMPLOYED_ANOMALY` flag (`src/preprocessing.py`). Leaving it in would put a fake number 1,000 years long into a numeric feature, while dropping it would lose the group's distinct risk level; the flag keeps that information. The same rows already carry `ORGANIZATION_TYPE = XNA`, which stays as its own category.

## 5. Outliers and skew in income

**Why:** income is central to affordability, and one glance at the maximum suggests an extreme value.

In [6]:
income = df["AMT_INCOME_TOTAL"]
print(income.describe().round(0))
print("skewness, raw:", round(income.skew(), 1), "| after log:", round(np.log(income).skew(), 2))
print("five largest:", income.nlargest(5).tolist())
print("99.9th percentile:", income.quantile(0.999), "| rows above it:", (income > income.quantile(0.999)).sum())
print("TARGET for the ten largest incomes:", df.loc[income.nlargest(10).index, "TARGET"].tolist())

count       307511.0
mean        168798.0
std         237123.0
min          25650.0
25%         112500.0
50%         147150.0
75%         202500.0
max      117000000.0
Name: AMT_INCOME_TOTAL, dtype: float64
skewness, raw: 391.6 | after log: 0.17
five largest: [117000000.0, 18000090.0, 13500000.0, 9000000.0, 6750000.0]
99.9th percentile: 900000.0 | rows above it: 278
TARGET for the ten largest incomes: [1, 0, 0, 0, 0, 0, 0, 0, 0, 0]


**Result and interpretation.** Median income is 147,150 but the maximum is 117,000,000, about 795 times the median. Skewness is 391.6 raw and 0.17 after a log transform, so the problem is real and measurable. A single applicant holds that maximum (and defaulted); the next largest income is 18 million, and 278 applicants sit above the 99.9th percentile of 900,000.

**Decision.**
- No rows are removed. Large incomes are plausible for a small number of applicants, nothing here is impossible, and deleting them would make the model blind to that region.
- A log transform is warranted for the logistic-regression baseline, where extreme values dominate a linear fit, and unnecessary for tree models, which only use the order of values. It will be applied inside the logistic pipeline and its effect compared before and after in Phase 5.

## 6. Impossible values and odd categories

In [7]:
print("non-positive income:", (income <= 0).sum())
print("DAYS_BIRTH >= 0:", (df["DAYS_BIRTH"] >= 0).sum(),
      "| DAYS_REGISTRATION > 0:", (df["DAYS_REGISTRATION"] > 0).sum(),
      "| DAYS_LAST_PHONE_CHANGE > 0:", (df["DAYS_LAST_PHONE_CHANGE"] > 0).sum())
print("age range (years):", (-df["DAYS_BIRTH"] / 365.25).min().round(1), "to", (-df["DAYS_BIRTH"] / 365.25).max().round(1))
print("CODE_GENDER == 'XNA':", (df["CODE_GENDER"] == "XNA").sum(),
      "| NAME_FAMILY_STATUS == 'Unknown':", (df["NAME_FAMILY_STATUS"] == "Unknown").sum())
print("constant columns:", [c for c in df.columns if df[c].nunique(dropna=False) == 1])
print()
print(df.groupby("NAME_INCOME_TYPE")["TARGET"].agg(["size", "mean"]).round(4))

non-positive income: 0
DAYS_BIRTH >= 0: 0 | DAYS_REGISTRATION > 0: 0 | DAYS_LAST_PHONE_CHANGE > 0: 0
age range (years): 20.5 to 69.1
CODE_GENDER == 'XNA': 4 | NAME_FAMILY_STATUS == 'Unknown': 2
constant columns: []

                        size    mean
NAME_INCOME_TYPE                    
Businessman               10  0.0000
Commercial associate   71617  0.0748
Maternity leave            5  0.4000
Pensioner              55362  0.0539
State servant          21703  0.0575
Student                   18  0.0000
Unemployed                22  0.3636
Working               158774  0.0959


**Result and interpretation.** No negative incomes, no impossible day counts, ages run from 20.5 to 69.1, and no column is constant. Four applicants have gender `XNA` and two have family status `Unknown`. Several income-type categories are tiny (for example 5 maternity-leave and 22 unemployed applicants), so their default rates are too unstable to interpret.

**Decision.** Keep all categories as they are. The 6 `XNA`/`Unknown` rows cannot affect a model trained on 300,000 rows, and merging rare income types would be a guess. Default rates for tiny groups will not be interpreted in EDA or in the segment analysis.

## 7. Leakage review of the application table

**Question for every column:** would this value be known when the loan decision is made?

Every column in this table comes from the application itself, according to the data owner's column descriptions, and only `TARGET` describes the outcome, so no post-outcome column is present. Two items need care, not removal:

- `DEF_30/60_CNT_SOCIAL_CIRCLE` describes defaults among people around the applicant. The description does not say when those defaults were observed. It will be kept for now and tested by removing it in Phase 5; a large drop in performance would need a closer look at timing.
- Aggregates built from the bureau and previous-application tables later must use only records dated before the application. In these tables all times are relative to the application date, so this can be enforced directly, and it will be checked when those tables are added.

## 8. Apply the cleaning rules

In [8]:
from src.preprocessing import clean_applications, split_features_target

X, y = split_features_target(clean_applications(df))
print("features:", X.shape, "| target:", y.shape)
print("anomaly flag rate:", round(X["DAYS_EMPLOYED_ANOMALY"].mean(), 4))
print("DAYS_EMPLOYED max after fix:", X["DAYS_EMPLOYED"].max())
print("SK_ID_CURR in features:", config.ID_COLUMN in X.columns)

features: (307511, 121) | target: (307511,)
anomaly flag rate: 0.1801
DAYS_EMPLOYED max after fix: 0.0
SK_ID_CURR in features: False


**Result.** 122 columns become 121 features: `TARGET` and the ID are removed and the anomaly flag is added. The largest `DAYS_EMPLOYED` value is now 0, a genuine one.

## Summary of decisions from this notebook

| Finding | Decision |
|---|---|
| ID unrelated to outcome | Exclude from features |
| Label = early payment difficulty, X and Y undisclosed | Word results accordingly; do not rely on accuracy |
| Missingness structural and sometimes informative | No blanket imputation; trees use missing values as they are; logistic imputation tested in Phase 5/6 |
| `DAYS_EMPLOYED` placeholder on 18.0% of rows | Set to missing and add flag |
| Income extremely skewed (391.6 raw, 0.17 log) | Keep all rows; log only where a linear model needs it, verified in Phase 5 |
| Tiny categories (gender XNA, unknown status, rare income types) | Keep; do not interpret their rates |
| Timing of social-circle features unclear | Keep; ablation test in Phase 5 |

## Next step

Phase 5: EDA with a question behind every chart, the statistical tests, and the ratio features (credit-to-income, annuity-to-income). The history tables are needed for the alternate-data question and have not been added yet.